# External Valuation
## March 5, 2026

# Data Dictionary

- id - unique ID for workout
- user_id - unique ID for athlete who did the workout
- sport_type - sport for the workout (~40 unique)
- sport_type_grouped - groups workouts into main groups
- speed_mph - miles / elapsed time in hours
- distance - distance in meters
- miles - distance in miles
- kilometers - distance in kilometers
- moving_time - seconds of active moving time (pauses for red light, water break, etc)
- elapsed_time - total seconds for entire workout
- moving_minutes - minutes of active moving time
- elapsed_minutes - minutes for entire workout
- moving_time_per - moving_minutes / elapsed_minutes
- total_elevation_gain - meters of climbing
- meters_per_km - avg meters of climbing per kilometer
- feet_per_mile - avg feet of climbing per mile (for the Americans lol)
- commute - boolean flag is user marked the activity as a commute (like when Oliver bikes to class)
- manual - flag for if the workout was generated by a tracking device or if user manually entered the details
- has_gear - boolean for if user indicated which shoes/bike they used for the workout
- suffer_score - Strava metric used to describe how tough the workout is; function of heart rate and total time
- kudos_count - how many "likes" the workout received on Strava
- device_name - name used to record the workout
- start_date - date of workout
- hour - hour of workout (start)
- day_part - morning vs afternoon vs evening vs night (start)
- month - month of workout
- dayofweek - day of week of workout
- is_weekend - boolean for if dayofweek == Saturday or Sunday
- is_northern_hemisphere - start_lat > 0
- num_turns - number of turns in the GPS trace
- turns_per_mile - num_turns / miles
- wobble - how wiggly vs straight the trace is (ignoring turns)
- sprawl - distance (in miles) from most northwest vs most southeast points in the trace
- is_winter - workout in Dec-Feb for northern hemisphere, or July-August for southern
- is_summer - workout in Dec-Feb for southern hemisphere, or July-August for northern

In [1]:
import joblib
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, log_loss

In [3]:
bundle = joblib.load("best_xgb_bundle_v2.joblib")
model = bundle["model"]
le = bundle["label_encoder"]
feature_cols = bundle["feature_cols"]
target_col = bundle["target_col"] 

In [4]:
df_new = pd.read_csv("data_for_modeling_Mar4.csv") 

In [5]:
df_new = df_new[df_new[target_col].notna()].copy()

train_labels = set(le.classes_)
new_labels = set(df_new[target_col].unique())
unseen_labels = new_labels - train_labels
print("Unseen labels in new data:", unseen_labels)

Unseen labels in new data: {'EBikeRide'}


In [6]:
df_new["sport_type_grouped"] = df_new["sport_type_grouped"].replace({
    "EBikeRide": "Ride"
})

In [7]:
y_new = le.transform(df_new["sport_type_grouped"])

In [8]:
# ----- Feature engineering -----
gps_cols = ["num_turns", "turns_per_mile", "wobble", "sprawl"]

# has_gps = 1 if num_turns is not NaN else 0
df_new["has_gps"] = df_new["num_turns"].notna().astype(int)

# fill NaNs in GPS-shape cols with 0
for c in gps_cols:
    if c in df_new.columns:
        df_new[c] = df_new[c].fillna(0)
    else:
        df_new[c] = 0

In [9]:
# month column preferred 
if "month" in df_new.columns:
    m = df_new["month"]
else:
    df_new["start_date_local"] = pd.to_datetime(df_new["start_date_local"], errors="coerce")
    m = df_new["start_date_local"].dt.month

df_new["is_winter"] = m.isin([12, 1, 2]).astype(int)
df_new["is_summer"] = m.isin([6, 7, 8]).astype(int)

In [10]:
# Drop leakage-prone columns in external data
leak_cols = ["kudos_count", "has_gear", "commute"]

to_drop = [c for c in leak_cols if c in df_new.columns]
if to_drop:
    print("Dropping leakage cols from df_new:", to_drop)
    df_new = df_new.drop(columns=to_drop).copy()
else:
    print("No leakage cols found in df_new.")

Dropping leakage cols from df_new: ['kudos_count', 'has_gear', 'commute']


In [11]:
# Align features to training schema
missing_cols = [c for c in feature_cols if c not in df_new.columns]
if missing_cols:
    raise ValueError(f"New data missing columns: {missing_cols}")

X_new = df_new[feature_cols].copy()

In [12]:
out_path = "data_for_modeling_Mar4_v2.csv"
df_new.to_csv(out_path, index=False)

print(f"Saved cleaned external dataset to: {out_path}")

Saved cleaned external dataset to: data_for_modeling_Mar4_v2.csv


In [13]:
missing_cols = [c for c in feature_cols if c not in df_new.columns]
if missing_cols:
    raise ValueError(f"New data missing columns: {missing_cols}")

X_new = df_new[feature_cols].copy()

In [14]:
y_pred = model.predict(X_new)

In [15]:
print("External Accuracy:", accuracy_score(y_new, y_pred))
print("External Macro F1:", f1_score(y_new, y_pred, average="macro"))
print(classification_report(y_new, y_pred))

External Accuracy: 0.885097192224622
External Macro F1: 0.7916532126253786
              precision    recall  f1-score   support

           0       0.44      0.43      0.43        49
           1       0.93      0.90      0.92       730
           2       0.88      0.94      0.91       860
           3       0.86      0.84      0.85       342
           4       0.90      0.80      0.85       334

    accuracy                           0.89      2315
   macro avg       0.80      0.78      0.79      2315
weighted avg       0.89      0.89      0.88      2315



In [16]:
y_true_label = le.inverse_transform(y_new)
y_pred_label = le.inverse_transform(y_pred)

result_df = df_new.copy()
result_df["y_true"] = y_true_label
result_df["y_pred"] = y_pred_label
result_df["correct"] = (result_df["y_true"] == result_df["y_pred"])

result_df[["y_true","y_pred","correct"]].head(20)

,y_true,y_pred,correct
0,Walk,Walk,True
1,Workout,Workout,True
2,Run,Run,True
3,Ride,Ride,True
4,Run,Run,True
5,Run,Run,True
6,Workout,Workout,True
7,Run,Run,True
8,Run,Run,True
9,Run,Run,True


In [17]:
labels = le.classes_ 
cm = confusion_matrix(y_new, y_pred)

cm_df = pd.DataFrame(cm, index=[f"True_{c}" for c in labels],
                        columns=[f"Pred_{c}" for c in labels])

cm_df

,Pred_Hike,Pred_Ride,Pred_Run,Pred_Walk,Pred_Workout
True_Hike,21,1,0,26,1
True_Ride,0,660,47,1,22
True_Run,5,29,812,11,3
True_Walk,18,3,29,288,4
True_Workout,4,16,36,10,268


In [18]:
errors = result_df[~result_df["correct"]]
errors.groupby(["y_true", "y_pred"]).size().sort_values(ascending=False).head(15)

y_true   y_pred 
Ride     Run        47
Workout  Run        36
Run      Ride       29
Walk     Run        29
Hike     Walk       26
Ride     Workout    22
Walk     Hike       18
Workout  Ride       16
Run      Walk       11
Workout  Walk       10
Run      Hike        5
Walk     Workout     4
Workout  Hike        4
Run      Workout     3
Walk     Ride        3
dtype: int64

## Model Performance Comparison

| Model                         | Accuracy | Macro F1 | Minority Class (Hike F1) | Overall Assessment |
|--------------------------------|----------|----------|---------------------------|--------------------|
| Logistic Regression_v1         | ~0.86    | ~0.74    | Low (~0.42 earlier)       | Underfits nonlinear patterns |
| HistGradientBoosting_v2        | 0.910    | 0.832    | 0.61                      | Comparable to HistGB |
| XGBoosting_v2                  | 0.915    | 0.842    | 0.64                      | Best balance & generalization|
| XGBoosting_validation          | 0.885    | 0.792    | 0.43                      | good generalization with expected external drop; weakness on Hike|